# 📈 US Tariff Stock Market Reaction Tracker — v2
### Research-Grade Data Science Project | Google Colab

**Objective:** Measure how US tariff announcements (2018–2025) shocked global stock markets using
event study methodology, FinBERT NLP, XGBoost prediction, and SHAP interpretability.

---
### 🆕 Upgrades in v2
- **SHAP Values** — explains *why* the XGBoost model made each prediction
- **Live NewsAPI** — fetches real headlines instead of hardcoded samples
- **International indices** — EWJ (Japan), FXI (China), EWG (Germany), EWC (Canada)

### 📦 Sections
1. Install & Import
2. Tariff Events
3. Stock Data — US Sectors + International Indices
4. CAPM Event Study → Abnormal Returns
5. EDA & Visualizations
6. Live News + FinBERT Sentiment
7. Correlation: Sentiment → Returns
8. XGBoost Model
9. SHAP Interpretability
10. International Contagion Analysis
11. Summary Dashboard

## ⚙️ Section 1 — Install & Import

In [ ]:
!pip install yfinance transformers torch plotly xgboost newsapi-python shap scikit-learn pandas numpy matplotlib seaborn -q
print('✅ All libraries installed')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
from transformers import pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import shap

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
from scipy import stats

# NewsAPI — only imported if key is provided
try:
    from newsapi import NewsApiClient
    NEWSAPI_AVAILABLE = True
except ImportError:
    NEWSAPI_AVAILABLE = False

print('✅ All imports successful')

## 📅 Section 2 — Tariff Events

In [ ]:
tariff_events = [
    {'date':'2018-01-22','event':'Solar & washing machine tariffs (20-50%)','country':'Global','sector':'Consumer Discretionary','tariff_pct':30},
    {'date':'2018-03-01','event':'Steel 25% & Aluminum 10% tariffs','country':'Global','sector':'Materials','tariff_pct':25},
    {'date':'2018-03-22','event':'$50B China tariff (Section 301)','country':'China','sector':'Technology','tariff_pct':25},
    {'date':'2018-06-15','event':'List 1: $34B China goods tariffed','country':'China','sector':'Industrials','tariff_pct':25},
    {'date':'2018-07-06','event':'List 1 tariffs take effect','country':'China','sector':'Industrials','tariff_pct':25},
    {'date':'2018-08-07','event':'List 2: $16B China goods','country':'China','sector':'Technology','tariff_pct':25},
    {'date':'2018-09-17','event':'List 3: $200B China goods at 10%','country':'China','sector':'Consumer Discretionary','tariff_pct':10},
    {'date':'2018-12-01','event':'US-China 90-day truce (G20)','country':'China','sector':'Technology','tariff_pct':0},
    {'date':'2019-05-05','event':'Threat to raise List 3 to 25%','country':'China','sector':'Consumer Discretionary','tariff_pct':25},
    {'date':'2019-05-10','event':'List 3 raised to 25%','country':'China','sector':'Consumer Discretionary','tariff_pct':25},
    {'date':'2019-08-01','event':'List 4A: 10% on $300B China goods','country':'China','sector':'Technology','tariff_pct':10},
    {'date':'2019-09-01','event':'List 4A takes effect','country':'China','sector':'Consumer Discretionary','tariff_pct':15},
    {'date':'2019-12-13','event':'Phase One deal agreed — rollback','country':'China','sector':'Technology','tariff_pct':-15},
    {'date':'2020-01-15','event':'Phase One deal signed','country':'China','sector':'Industrials','tariff_pct':-7},
    {'date':'2025-01-20','event':'Executive order reviewing trade deals','country':'Global','sector':'Industrials','tariff_pct':10},
    {'date':'2025-02-01','event':'25% tariffs on Canada & Mexico','country':'Canada/Mexico','sector':'Energy','tariff_pct':25},
    {'date':'2025-02-04','event':'10% tariffs on China','country':'China','sector':'Technology','tariff_pct':10},
    {'date':'2025-03-04','event':'Canada/Mexico tariffs take effect','country':'Canada/Mexico','sector':'Consumer Staples','tariff_pct':25},
    {'date':'2025-04-02','event':'Liberation Day — sweeping global tariffs','country':'Global','sector':'Consumer Discretionary','tariff_pct':10},
    {'date':'2025-04-09','event':'90-day pause on most tariffs','country':'Global','sector':'Technology','tariff_pct':-10},
    {'date':'2025-04-11','event':'China tariffs raised to 145%','country':'China','sector':'Technology','tariff_pct':145},
    {'date':'2025-05-12','event':'US-China truce — tariffs to 30%','country':'China','sector':'Technology','tariff_pct':-115},
    {'date':'2025-06-04','event':'50% tariff on EU threatened','country':'EU','sector':'Consumer Discretionary','tariff_pct':50},
    {'date':'2025-07-09','event':'Reciprocal pause extended 90 days','country':'Global','sector':'Industrials','tariff_pct':0},
]

events_df = pd.DataFrame(tariff_events)
events_df['date'] = pd.to_datetime(events_df['date'])
events_df['is_escalation'] = events_df['tariff_pct'] > 0
print(f'✅ {len(events_df)} tariff events | {events_df["is_escalation"].sum()} escalations | {(~events_df["is_escalation"]).sum()} de-escalations')
events_df.head(8)

## 📊 Section 3 — Stock Data: US Sectors + International Indices

In [ ]:
# US Sector ETFs
US_SECTORS = {
    'XLK':'Technology','XLI':'Industrials','XLB':'Materials',
    'XLY':'Consumer Discretionary','XLP':'Consumer Staples',
    'XLE':'Energy','XLF':'Financials','XLV':'Healthcare',
    'SPY':'S&P 500 (Market)',
}

# 🆕 International indices — this is the key upgrade for geographic scope
INTL_INDICES = {
    'EWJ':'Japan (Nikkei)',
    'FXI':'China (CSI 300)',
    'EWG':'Germany (DAX)',
    'EWC':'Canada (TSX)',
    'EWA':'Australia (ASX)',
    'EWU':'UK (FTSE 100)',
    'MCHI':'MSCI China',
    'VWO':'Emerging Markets',
}

ALL_TICKERS = {**US_SECTORS, **INTL_INDICES}

print(f'📥 Downloading data for {len(ALL_TICKERS)} tickers (US sectors + international)...')
raw = yf.download(
    tickers=list(ALL_TICKERS.keys()),
    start='2017-01-01', end='2025-12-31',
    auto_adjust=True, progress=False
)

prices  = raw['Close'].copy()
prices.columns = [ALL_TICKERS[t] for t in prices.columns]
returns = prices.pct_change().dropna()

# Separate US and international
us_cols   = list(US_SECTORS.values())
intl_cols = list(INTL_INDICES.values())

print(f'✅ {len(prices)} trading days | {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'   US sectors: {len(us_cols)} | International: {len(intl_cols)}')
prices.tail(3)

## 📐 Section 4 — CAPM Event Study

In [ ]:
def compute_abnormal_returns(event_date, ticker_col, returns_df, window=(-5,10), estimation=(-30,-2)):
    market_col = 'S&P 500 (Market)'
    idx = returns_df.index.searchsorted(event_date)
    if idx >= len(returns_df): return None
    es, ee = max(0,idx+estimation[0]), max(0,idx+estimation[1])
    evs, eve = max(0,idx+window[0]), min(len(returns_df)-1, idx+window[1])
    if ee <= es or ticker_col not in returns_df.columns: return None
    est = returns_df.iloc[es:ee]
    model = LinearRegression().fit(est[market_col].values.reshape(-1,1), est[ticker_col].values)
    alpha, beta = model.intercept_, model.coef_[0]
    ev  = returns_df.iloc[evs:eve+1]
    ar  = ev[ticker_col] - (alpha + beta * ev[market_col])
    return {
        'event_date':event_date, 'ticker':ticker_col,
        'alpha':round(alpha,6), 'beta':round(beta,4),
        'AR_mean':round(ar.mean()*100,4),
        'CAR':round(ar.sum()*100,4),
        'AR_series':(ar*100).values.tolist(),
        'AR_dates':ev.index.tolist(), 'n_days':len(ar)
    }

print('🔄 Running event study across all tickers and events...')
results = []
for _, row in events_df.iterrows():
    for ticker_col in list(us_cols) + list(intl_cols):
        if ticker_col == 'S&P 500 (Market)': continue
        res = compute_abnormal_returns(row['date'], ticker_col, returns)
        if res:
            res.update({'event':row['event'],'country':row['country'],
                        'tariff_pct':row['tariff_pct'],'is_escalation':row['is_escalation'],
                        'is_us': ticker_col in us_cols})
            results.append(res)

results_df = pd.DataFrame(results).dropna(subset=['CAR'])
results_df['year'] = results_df['event_date'].dt.year
results_df['month'] = results_df['event_date'].dt.month
results_df['ticker_encoded'] = pd.Categorical(results_df['ticker']).codes

print(f'✅ {len(results_df)} ticker-event pairs | US: {results_df["is_us"].sum()} | Intl: {(~results_df["is_us"]).sum()}')
results_df[['event_date','ticker','beta','CAR','tariff_pct','is_escalation','is_us']].head(10)

## 📊 Section 5 — EDA & Visualizations

In [ ]:
# Plot 1: Normalised price performance
norm = (prices / prices.iloc[0]) * 100
fig = go.Figure()
colors_p = px.colors.qualitative.Set2
for i, col in enumerate(norm.columns):
    lw   = 3 if col == 'S&P 500 (Market)' else 1.5
    dash = 'dash' if col in intl_cols else 'solid'
    fig.add_trace(go.Scatter(x=norm.index,y=norm[col],name=col,
                             line=dict(width=lw,color=colors_p[i%len(colors_p)],dash=dash)))
for _,ev in events_df.iterrows():
    fig.add_vline(x=ev['date'],line_width=1,line_dash='dot',
                  line_color='red' if ev['is_escalation'] else 'green',opacity=0.35)
fig.update_layout(title='US Sectors + International Indices (Normalised) with Tariff Events',
                  height=520,template='plotly_white',hovermode='x unified')
fig.show()
print('Solid lines = US sectors | Dashed lines = International indices')

In [ ]:
# Plot 2: Average CAR by ticker during escalations
esc = results_df[results_df['is_escalation']]
avg_car = esc.groupby(['ticker','is_us'])['CAR'].mean().reset_index().sort_values('CAR')

fig, ax = plt.subplots(figsize=(14,8))
bar_colors = ['#185FA5' if r else '#D85A30' for r in avg_car['is_us']]
bars = ax.barh(avg_car['ticker'], avg_car['CAR'], color=bar_colors, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
for bar, val in zip(bars, avg_car['CAR']):
    ax.text(val+(0.05 if val>=0 else -0.05), bar.get_y()+bar.get_height()/2,
            f'{val:.2f}%', va='center', ha='left' if val>=0 else 'right', fontsize=8)
us_patch   = mpatches.Patch(color='#185FA5', label='US Sector')
intl_patch = mpatches.Patch(color='#D85A30', label='International Index')
ax.legend(handles=[us_patch, intl_patch], loc='lower right')
ax.set_title('Average CAR (%) During Tariff Escalations — US vs International', fontsize=13, fontweight='bold')
ax.set_xlabel('Average Cumulative Abnormal Return (%)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('car_us_vs_intl.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 3: Heatmap — ticker x year
heat = results_df.groupby(['ticker','year'])['CAR'].mean().unstack(fill_value=0)
plt.figure(figsize=(14,9))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.5, cbar_kws={'label':'Avg CAR (%)'})
plt.title('Average CAR (%) by Ticker and Year', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Year'); plt.ylabel('Ticker')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('heatmap_ticker_year.png', dpi=150, bbox_inches='tight')
plt.show()

## 📰 Section 6 — Live News + FinBERT Sentiment
### 🆕 NewsAPI Integration
Get a free API key at **newsapi.org** (100 requests/day free). If you don't have one, the notebook falls back to the built-in headline dataset.

In [ ]:
# ─── CONFIG: paste your NewsAPI key here ────────────────────────────────────
NEWSAPI_KEY = ''   # e.g. 'abc123def456...'
# ────────────────────────────────────────────────────────────────────────────

FALLBACK_HEADLINES = [
    {'date':'2018-03-01','headline':'Trump announces steep tariffs on steel and aluminum imports, rattling global markets'},
    {'date':'2018-03-22','headline':'United States to impose 25 percent tariff on 50 billion dollars of Chinese goods'},
    {'date':'2018-07-06','headline':'Trade war begins as US tariffs on Chinese goods take effect, China retaliates immediately'},
    {'date':'2018-09-17','headline':'Trump escalates trade war with tariffs on 200 billion dollars in Chinese products'},
    {'date':'2019-05-05','headline':'Trump threatens to raise tariffs on China to 25 percent amid stalled trade talks'},
    {'date':'2019-05-10','headline':'US raises tariffs to 25 percent, stock market plunges on trade war fears'},
    {'date':'2019-08-01','headline':'Trump announces new tariffs on 300 billion in Chinese goods sending markets lower'},
    {'date':'2019-12-13','headline':'Phase One trade deal agreed, stocks rally sharply on tariff rollback news'},
    {'date':'2020-01-15','headline':'US and China sign Phase One trade deal in White House ceremony'},
    {'date':'2025-02-01','headline':'Trump slaps 25 percent tariffs on Canada and Mexico, threatening supply chains'},
    {'date':'2025-04-02','headline':'Liberation Day tariffs shock global markets as Trump imposes sweeping new duties'},
    {'date':'2025-04-09','headline':'Trump pauses most tariffs for 90 days, global markets surge on relief'},
    {'date':'2025-04-11','headline':'US raises China tariffs to 145 percent in largest escalation of trade war'},
    {'date':'2025-05-12','headline':'US and China reach trade truce, tariffs slashed from 145 to 30 percent'},
    {'date':'2025-06-04','headline':'Trump threatens 50 percent tariff on EU goods, European markets tumble'},
    {'date':'2018-12-01','headline':'US and China agree to trade truce at G20, markets surge on deal hopes'},
    {'date':'2019-09-01','headline':'New tariffs on consumer goods take effect with mixed market reaction'},
    {'date':'2025-07-09','headline':'US extends tariff pause for additional 90 days as trade talks continue'},
    {'date':'2018-08-07','headline':'Second round of tariffs on China hits semiconductors and electronics sectors'},
    {'date':'2025-02-04','headline':'China tariffs take effect as trade tensions escalate between world powers'},
]

def fetch_live_headlines(api_key, queries=None, from_date='2018-01-01', max_per_query=100):
    """
    Fetch real tariff headlines from NewsAPI.
    Returns a DataFrame with date and headline columns.
    """
    if queries is None:
        queries = ['US tariffs China', 'US tariffs trade war', 'Trump tariffs stock market',
                   'tariff retaliation', 'trade war escalation', 'US tariffs Canada Mexico']
    newsapi = NewsApiClient(api_key=api_key)
    articles = []
    for q in queries:
        try:
            resp = newsapi.get_everything(
                q=q, language='en',
                from_param=from_date,
                sort_by='relevancy',
                page_size=min(max_per_query, 100)
            )
            for art in resp.get('articles', []):
                if art.get('title') and art.get('publishedAt'):
                    articles.append({
                        'date': art['publishedAt'][:10],
                        'headline': art['title'],
                        'source': art.get('source', {}).get('name', '')
                    })
            print(f'  Fetched {len(resp.get("articles", []))} articles for: "{q}"')
        except Exception as e:
            print(f'  Error fetching "{q}": {e}')
    df = pd.DataFrame(articles).drop_duplicates(subset='headline')
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values('date').reset_index(drop=True)

# Load headlines
if NEWSAPI_KEY and NEWSAPI_AVAILABLE:
    print('🌐 Fetching live headlines from NewsAPI...')
    headlines_df = fetch_live_headlines(NEWSAPI_KEY)
    print(f'✅ Fetched {len(headlines_df)} unique live headlines')
else:
    print('ℹ️  No NewsAPI key found — using built-in headline dataset')
    print('   To use live news: set NEWSAPI_KEY above (free at newsapi.org)')
    headlines_df = pd.DataFrame(FALLBACK_HEADLINES)
    headlines_df['date'] = pd.to_datetime(headlines_df['date'])
    print(f'✅ Loaded {len(headlines_df)} built-in headlines')

headlines_df.head()

In [ ]:
# Load FinBERT and run sentiment analysis
print('📥 Loading FinBERT (first run ~2 min)...')
finbert = pipeline('text-classification', model='ProsusAI/finbert',
                   tokenizer='ProsusAI/finbert', device=-1)
print('✅ FinBERT loaded')

def analyze_sentiment(text):
    r = finbert(text[:400])[0]
    label   = r['label'].lower()
    score   = r['score']
    numeric = {'positive':1,'neutral':0,'negative':-1}.get(label,0) * score
    return pd.Series({'sentiment_label':label,'sentiment_score':score,'sentiment_numeric':numeric})

print(f'🔍 Analyzing sentiment for {len(headlines_df)} headlines...')
sent_cols = headlines_df['headline'].apply(analyze_sentiment)
headlines_df = pd.concat([headlines_df, sent_cols], axis=1)

print('\n✅ Sentiment distribution:')
print(headlines_df['sentiment_label'].value_counts())
headlines_df[['date','headline','sentiment_label','sentiment_numeric']].head(8)

In [ ]:
# Sentiment over time — interactive chart
color_map = {'positive':'#2ca02c','neutral':'#ff7f0e','negative':'#d62728'}
fig = make_subplots(rows=2, cols=1,
                    subplot_titles=['FinBERT Sentiment Score Over Time','Distribution'],
                    row_heights=[0.65,0.35], vertical_spacing=0.15)
for label, grp in headlines_df.groupby('sentiment_label'):
    fig.add_trace(go.Scatter(
        x=grp['date'], y=grp['sentiment_numeric'], mode='markers',
        marker=dict(size=8, color=color_map[label]),
        name=label.capitalize(),
        text=grp['headline'],
        hovertemplate='<b>%{text}</b><br>Score: %{y:.3f}<extra></extra>'
    ), row=1, col=1)
counts = headlines_df['sentiment_label'].value_counts()
fig.add_trace(go.Bar(
    x=counts.index, y=counts.values,
    marker_color=[color_map.get(l,'gray') for l in counts.index],
    showlegend=False
), row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=1)
fig.update_layout(title='FinBERT Sentiment on Tariff Headlines', height=600, template='plotly_white')
fig.show()

## 🔗 Section 7 — Sentiment → Returns Correlation

In [ ]:
# Merge sentiment with event study results
merged = pd.merge(
    results_df[['event_date','ticker','CAR','AR_mean','beta','tariff_pct','is_escalation','is_us']],
    headlines_df[['date','sentiment_numeric','sentiment_label']],
    left_on='event_date', right_on='date', how='inner'
).drop(columns='date')

print(f'✅ Merged: {len(merged)} event-headline pairs')

# Correlation matrix
corr = merged[['sentiment_numeric','CAR','tariff_pct','beta']].corr()
plt.figure(figsize=(7,5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink':0.8})
plt.title('Correlation Matrix', fontsize=13, pad=10)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n Sentiment ↔ CAR correlation: {corr.loc["sentiment_numeric","CAR"]:.3f}')

In [ ]:
# Regression scatter: sentiment vs CAR
if len(merged) >= 5:
    x = merged['sentiment_numeric'].values
    y = merged['CAR'].values
    slope, intercept, r_val, p_val, _ = stats.linregress(x, y)

    fig, ax = plt.subplots(figsize=(9,6))
    colors_s = merged['is_us'].map({True:'#185FA5', False:'#D85A30'})
    ax.scatter(x, y, c=colors_s, s=70, alpha=0.8, zorder=5)
    xl = np.linspace(x.min(), x.max(), 100)
    ax.plot(xl, slope*xl+intercept, 'k--', lw=2, label=f'Fit (R²={r_val**2:.3f}, p={p_val:.3f})')
    ax.axhline(0, color='gray', lw=0.8, ls=':')
    ax.axvline(0, color='gray', lw=0.8, ls=':')
    us_p   = mpatches.Patch(color='#185FA5', label='US Sector')
    intl_p = mpatches.Patch(color='#D85A30', label='International')
    ax.legend(handles=[us_p, intl_p, ax.lines[0]])
    ax.set_xlabel('FinBERT Sentiment Score')
    ax.set_ylabel('Cumulative Abnormal Return (%)')
    ax.set_title('Sentiment Score vs CAR — US & International Markets', fontsize=13)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('sentiment_vs_car.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'R² = {r_val**2:.4f} | p = {p_val:.4f}')

## 🤖 Section 8 — XGBoost Model

In [ ]:
ml_df = results_df.copy()
ml_df['ticker_encoded'] = pd.Categorical(ml_df['ticker']).codes
features = ['tariff_pct','beta','is_escalation','ticker_encoded','year','month','AR_mean','is_us']
target   = 'CAR'
ml_df = ml_df.dropna(subset=features+[target])
X = ml_df[features].astype(float)
y = ml_df[target].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_te_s = scaler.transform(X_test)

xgb_model = xgb.XGBRegressor(n_estimators=150, max_depth=4, learning_rate=0.08,
                               subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
xgb_model.fit(X_tr_s, y_train)
y_pred = xgb_model.predict(X_te_s)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'🤖 XGBoost  RMSE={rmse:.4f}%  R²={r2:.4f}')

fig, axes = plt.subplots(1,2,figsize=(13,5))
imp = pd.Series(xgb_model.feature_importances_, index=features).sort_values()
imp.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('XGBoost Feature Importance', fontweight='bold')
axes[0].set_xlabel('Importance')
lims = [min(y_test.min(),y_pred.min()), max(y_test.max(),y_pred.max())]
axes[1].scatter(y_test, y_pred, alpha=0.6, color='darkorange', s=50)
axes[1].plot(lims, lims, 'r--', lw=1.5)
axes[1].set_xlabel('Actual CAR (%)'); axes[1].set_ylabel('Predicted CAR (%)')
axes[1].set_title(f'Actual vs Predicted  R²={r2:.3f}', fontweight='bold')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('xgboost_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔍 Section 9 — SHAP Interpretability
### 🆕 This is the key upgrade — explains WHY the model predicts what it does

SHAP (SHapley Additive exPlanations) decomposes every single prediction into the contribution
of each feature. Instead of just knowing the model works, you can explain *why* a specific tariff
event caused a large negative CAR in the Technology sector.

In [ ]:
print('🔍 Computing SHAP values...')
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_te_s)
print(f'✅ SHAP values computed for {len(X_te_s)} test samples')

# SHAP Summary Plot — shows overall feature importance AND direction
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_te_s,
    feature_names=features,
    plot_type='dot',
    show=False
)
plt.title('SHAP Summary Plot — Feature Impact on CAR Prediction', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('How to read: Right side (pink) = pushes CAR higher | Left side (blue) = pushes CAR lower')
print('Dot colour = feature value (red=high, blue=low)')

In [ ]:
# SHAP Bar plot — mean absolute impact
plt.figure(figsize=(9, 5))
shap.summary_plot(
    shap_values, X_te_s,
    feature_names=features,
    plot_type='bar',
    show=False
)
plt.title('Mean |SHAP| — Average Feature Contribution to Predictions', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Waterfall — explain a single prediction
# Pick the event with the largest negative predicted CAR (most dramatic tariff shock)
worst_idx = np.argmin(y_pred)

print(f'Explaining prediction #{worst_idx} — the most negative CAR prediction:')
print(f'  Predicted CAR: {y_pred[worst_idx]:.2f}%')
print(f'  Actual CAR:    {y_test.iloc[worst_idx]:.2f}%')
print(f'  Features:      {dict(zip(features, X_test.iloc[worst_idx].round(3)))}')

plt.figure(figsize=(10, 6))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[worst_idx],
        base_values=explainer.expected_value,
        data=X_te_s[worst_idx],
        feature_names=features
    ),
    show=False
)
plt.title(f'SHAP Waterfall — Why did this event predict CAR={y_pred[worst_idx]:.2f}%?', fontsize=12, pad=12)
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Waterfall shows how each feature pushed the prediction UP or DOWN from the baseline')

In [ ]:
# SHAP Dependence Plot — how tariff_pct interacts with beta
plt.figure(figsize=(9, 5))
shap.dependence_plot(
    'tariff_pct', shap_values, X_te_s,
    feature_names=features,
    interaction_index='beta',
    show=False
)
plt.title('SHAP Dependence — tariff_pct vs SHAP value (coloured by beta)', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Shows: larger tariffs → more negative SHAP impact. High-beta tickers (red) are hit harder.')

## 🌍 Section 10 — International Contagion Analysis
### 🆕 Did US tariffs hurt foreign markets more than US markets?

In [ ]:
# Compare average CAR: US sectors vs international indices, by event type
contagion = results_df.groupby(['is_us','is_escalation'])['CAR'].agg(['mean','std','count']).reset_index()
contagion['label'] = contagion.apply(
    lambda r: ('US Sectors' if r['is_us'] else 'International') +
              ' — ' + ('Escalation' if r['is_escalation'] else 'De-escalation'), axis=1)
print('Average CAR comparison:')
print(contagion[['label','mean','std','count']].to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
bar_cols = ['#d62728','#1a9850','#d68728','#4a90d9']
bars = ax.bar(contagion['label'], contagion['mean'], color=bar_cols, edgecolor='white')
ax.errorbar(contagion['label'], contagion['mean'], yerr=contagion['std'],
            fmt='none', color='black', capsize=5, linewidth=1.2)
ax.axhline(0, color='black', lw=0.8, ls='--')
for bar, val in zip(bars, contagion['mean']):
    ax.text(bar.get_x()+bar.get_width()/2, val+(0.1 if val>=0 else -0.15),
            f'{val:.2f}%', ha='center', va='bottom' if val>=0 else 'top', fontsize=9)
ax.set_title('US Tariff Contagion — US Sectors vs International Indices', fontsize=13, fontweight='bold')
ax.set_ylabel('Average CAR (%)')
ax.set_xlabel('')
plt.xticks(rotation=15, ha='right')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('contagion_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Country-level reaction heatmap for international indices
intl_df = results_df[~results_df['is_us']].copy()
intl_heat = intl_df.groupby(['ticker','year'])['CAR'].mean().unstack(fill_value=0)

plt.figure(figsize=(12, 6))
sns.heatmap(intl_heat, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.5, cbar_kws={'label':'Avg CAR (%)'})
plt.title('International Market Reactions to US Tariff Events by Year', fontsize=13, fontweight='bold')
plt.xlabel('Year'); plt.ylabel('Index')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('intl_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key question: Which country suffered most from US tariff shocks?')

## 📋 Section 11 — Final Summary

In [ ]:
print('='*65)
print(' US TARIFF STOCK MARKET REACTION TRACKER v2 — SUMMARY')
print('='*65)
print(f'\n Events analyzed:        {len(events_df)}')
print(f' Headlines analyzed:     {len(headlines_df)}')
print(f' Tickers tracked:        {len(ALL_TICKERS)} ({len(us_cols)} US + {len(intl_cols)} International)')
print(f' Event-ticker pairs:     {len(results_df)}')

worst_us   = results_df[results_df['is_us'] & results_df['is_escalation']].groupby('ticker')['CAR'].mean().idxmin()
worst_intl = results_df[~results_df['is_us'] & results_df['is_escalation']].groupby('ticker')['CAR'].mean().idxmin()
best_intl  = results_df[~results_df['is_us'] & results_df['is_escalation']].groupby('ticker')['CAR'].mean().idxmax()

print(f'\n Most hurt US sector:     {worst_us}')
print(f' Most hurt intl index:    {worst_intl}')
print(f' Most resilient intl:     {best_intl}')
print(f'\n XGBoost  R²={r2:.4f}  RMSE={rmse:.4f}%')

sent_car = merged[['sentiment_numeric','CAR']].corr().iloc[0,1]
print(f' Sentiment↔CAR corr:     {sent_car:.3f}')
print(f'\n SHAP plots saved:        shap_summary.png, shap_bar.png')
print(f'                          shap_waterfall.png, shap_dependence.png')
print('='*65)